核心教学内容

- 多模型与兼容性（Multi-model Support）：
  - 展示如何通过 OpenAI 兼容的 API 接口（如 OpenRouter, Groq, Google Gemini），仅通过修改基地址（Base URL）和客户端配置，轻松切换底层的 LLM。
  - 重点说明了 Agent 框架不应绑定单一模型，而是具备良好的模型抽象能力。
- 结构化输出（Structured Outputs）：
  - 引入 Pydantic 框架，演示如何定义一个 Python 对象（BaseModel）来作为 LLM 的预期输出格式。
  - 教授如何通过让 LLM 输出 JSON 并配合模式（Schema）约束，将非结构化的自然语言文本“映射”为强类型的 Python 对象。
- 防线机制（Guardrails）：
  - 探讨如何通过代码逻辑或额外的 LLM 调用来监控 Agent 的行为，防止其输出不良内容（如不专业、包含占位符等）。
  - 区分了三种防线：输入校验、输出校验和工具调用校验。
- 进阶：Sandbox Agents 与 MCP（选修）：
  - 引入了“持久化工作区”概念，允许 Agent 在沙箱环境中搜索文档、编辑文件、执行命令并保存状态。
  - 预告了 MCP（Model Context Protocol）的使用，展示 Agent 如何通过标准协议连接外部上下文。

教学目标

- 从“Demo 走向工程化”：让学生意识到，真实的 AI Agent 项目不能只靠提示词，必须要有“结构化约束”来保证输出格式的统一，要有“防线”来保证业务质量。
- 掌握“类型安全”开发模式：通过使用 Pydantic 和结构化输出，让 AI 的返回结果在代码中是可控、可调用的对象，而不是仅仅是一串无法解析的文本。
- 构建健壮的 Agent 流水线：理解如何通过不同智能体的协作（比如一个负责写邮件，一个负责审核邮件）来构建防错机制，这是构建复杂 Agent 的基础设计原则。

In [24]:
import sys
from pathlib import Path
import os

try:
    # .py 脚本
    current_dir = Path(__file__).resolve().parent
except NameError:
    # .ipynb
    current_dir = Path.cwd()

project_root = current_dir.parent
sys.path.append(str(project_root))

import logging
from config.log_config import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

# 检查 api key

api_configs = {
    "deepseek_api_key": "DEEPSEEK_API_KEY",
    "bigmodel_api_key": "BIGMODEL_API_KEY",
    "moonshot_api_key": "MOONSHOT_API_KEY",
    "gemini_api_key": "GEMINI_API_KEY",
    "qianfan_api_key": "QIANFAN_API_KEY",
    "dashscope_api_key": "DASHSCOPE_API_KEY",
}

for name, env in api_configs.items():
    key = os.getenv(env)
    if key:
        logger.info(f"{name}: {key[:10]}")
    else:
        raise RuntimeError(f"{env} is not set")


DASHSCOPE_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
QIANFAN_BASE_URL = "https://qianfan.baidubce.com/v2"
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MOONSHOT_BASE_URL = "https://api.moonshot.cn/v1"
BIGMODEL_BASE_URL = "https://open.bigmodel.cn/api/paas/v4"

2026-06-28 21:26:47,845 [INFO] __main__: deepseek_api_key: sk-5837625
2026-06-28 21:26:47,848 [INFO] __main__: bigmodel_api_key: 50b41d629f
2026-06-28 21:26:47,849 [INFO] __main__: moonshot_api_key: sk-0e1imqn
2026-06-28 21:26:47,849 [INFO] __main__: gemini_api_key: AIzaSyDpOi
2026-06-28 21:26:47,849 [INFO] __main__: qianfan_api_key: bce-v3/ALT
2026-06-28 21:26:47,850 [INFO] __main__: dashscope_api_key: sk-3789376


In [34]:
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    trace,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    function_tool,
)

set_tracing_disabled(True)

#################### clien ####################
qwen_client = AsyncOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"), base_url=DASHSCOPE_BASE_URL
)

zhipu_client = AsyncOpenAI(
    api_key=os.getenv("BIGMODEL_API_KEY"), base_url=BIGMODEL_BASE_URL
)

qianfan_client = AsyncOpenAI(
    api_key=os.getenv("QIANFAN_API_KEY"), base_url=QIANFAN_BASE_URL
)

#################### model ####################
qwen_model = OpenAIChatCompletionsModel(model="qwen3.7-plus", openai_client=qwen_client)
zhipu_model = OpenAIChatCompletionsModel(model="glm-5.1", openai_client=zhipu_client)
qianfan_model = OpenAIChatCompletionsModel(
    model="ernie-4.5-turbo-128k", openai_client=qianfan_client
)

#################### agent ####################
sales_instructions = """你是一名 ComplAI 公司的销售代理。
ComplAI 是一家提供 SaaS 工具的公司，旨在通过 AI 技术助力企业确保 SOC2 合规并做好审计准备。
你的任务是撰写极具吸引力的销售邮件，以最大程度地提高客户的回复率。"""

qwen_sales_agent = Agent(
    name="qwen sales agent", instructions=sales_instructions, model=qwen_model
)
zhipu_sales_agent = Agent(
    name="qwen sales agent", instructions=sales_instructions, model=zhipu_model
)
qianfan_sales_agent = Agent(
    name="qwen sales agent", instructions=sales_instructions, model=qianfan_model
)

#################### agent tool ####################
tool_description = (
    "使用此工具撰写销售邮件。在 input 参数中，只需指令它‘撰写一封销售邮件’即可"
)

qwen_sales_tool = qwen_sales_agent.as_tool(
    tool_name="qwen sales tool", tool_description=tool_description
)
zhipu_sales_tool = zhipu_sales_agent.as_tool(
    tool_name="zhipu sales tool", tool_description=tool_description
)
qianfan_sales_tool = qianfan_sales_agent.as_tool(
    tool_name="qianfan sales tool", tool_description=tool_description
)

#################### function tool ####################

USE_EMAIL = True


def send_message(subject: str, text_body: str, html_body: str):
    if USE_EMAIL:
        print(
            f"send email: \n\nsubject | {subject[:15]}, \n\ntext_body | {text_body[:30]}, \n\nhtml_body | {html_body[:30]}\n\n"
        )
    else:
        print(
            f"push: subject | \n\n{subject[:15]}, \n\ntext_body | {text_body[:30]}, \n\nhtml_body | {html_body[:30]}\n\n"
        )


# 这里函数内部的注释, 使用 tool 的 agent 也会看到, 它会基于注释做决策
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str):
    """
    基于给定的邮件主题和正文, 给所有的潜在销售机会发送一封邮件

    Args:
        subject: 邮件的主题
        text_body: 邮件的纯文本正文
        html_body: 邮件的 html 格式的正文
    """

    send_message(subject, text_body, html_body)


tools = [qwen_sales_tool, zhipu_sales_tool, qianfan_sales_tool, send_email_tool]

In [26]:
#################### start agent ####################

# 这里用了云百炼上的 deepseek 模型
deepseek_model = OpenAIChatCompletionsModel(
    model="deepseek-v4-pro", openai_client=qwen_client
)

# agent 可以通过 tool 的 tool description 判断哪些 tool 是 sales writer tool
sales_manager_instructions = """你是一名 ComplAI 公司的销售经理。你的目标是通过使用 sales writer 工具，筛选出一封最出色的销售冷启动邮件。"""

sales_manager_agent = Agent(
    name="sales manager agent",
    instructions=sales_manager_instructions,
    tools=tools,
    model=deepseek_model,
)

In [27]:
#################### 启动 ####################
task = """请按照以下步骤执行：
生成草稿：使用三个 sales email writer 工具分别生成不同的邮件草稿。只需指令它们撰写销售邮件即可，无需提供额外细节。在三个工具各自完成草稿之前，请勿进入下一步。
评估与挑选：审阅这些草稿，根据你的专业判断，选出其中转化效果最好的一封。
发送邮件：使用你的工具将选出的那一封（且仅限那一封）最优秀的邮件发送给用户。只能发送 1 封邮件。"""

result = await Runner.run(starting_agent=sales_manager_agent, input=task)

logger.info(result.final_output)

send email: 

subject | 告别 SOC2 审计噩梦 → , 

text_body | Hi，

每次提到 SOC2 审计，大多数 SaaS 团队的, 

html_body | <html>
<body style="font-famil




2026-06-28 21:28:32,074 [INFO] __main__: ---

## ✅ 任务完成！总结如下：

| 步骤 | 详情 |
|------|------|
| **生成草稿** | 同时调用 Qwen、Zhipu、Qianfan 三个工具，各生成了一封销售邮件草稿 |
| **评估挑选** | 从痛点共鸣、价值主张、CTA 低摩擦度、真诚度、专业度五个维度综合评估，**Zhipu 的邮件以最高分胜出** |
| **发送邮件** | 已通过 `send_email_tool` 将 Zhipu 生成的最佳邮件发送给所有潜在客户 |

### 🏆 获胜邮件亮点：

- **痛点共鸣开场**：直接描绘 SOC2 审计的痛苦场景，瞬间拉近距离
- **清晰价值主张**：80% 时间缩短 + 三大核心功能，一目了然
- **极低摩擦 CTA**：「只需回复'暂无'，我保证不再打扰」——这是冷启动邮件中转化率最高的技巧之一
- **社会认同**：引用真实客户案例（3 周完成 3 个月的工作）
- **真诚收尾**：不施压，尊重对方时间，反而更容易获得回复


$\color{yellow}{结构化输出}$

LLM 默认输出的是自然语言, 但是我们可以让它输出 python object

In [28]:
from pydantic import BaseModel, Field

In [29]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="邮件是否专业, 得体")
    number_of_sentences: int = Field(
        description="除问候语和签名外, 邮件正文所包括的句子的数量"
    )
    is_contains_placeholders: bool = Field(
        description="邮件中是否包含任何形如 [xxx] 或 【xxx】 的待替换个性化占位符"
    )


EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': '邮件是否专业, 得体',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': '除问候语和签名外, 邮件正文所包括的句子的数量',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'is_contains_placeholders': {'description': '邮件中是否包含任何形如 [xxx] 或 【xxx】 的待替换个性化占位符',
   'title': 'Is Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'is_contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [30]:
checker_agent = Agent(
    name="checker agent",
    instructions="检查潜在机会的销售邮件",
    output_type=EmailReview,
    model=deepseek_model,
)

email = """
Hi [first_name],

与您联系是想看您是否愿意购买我们的产品。真的很棒。如果不买，您之后会错过的。

Devin
"""

result = await Runner.run(starting_agent=checker_agent, input=email)

logger.info(result.final_output)

2026-06-28 21:28:34,822 [INFO] __main__: is_professional=False number_of_sentences=3 is_contains_placeholders=True


$\color{yellow}{安全防护措施}$

agents SDK 中非常典型的防御性编程（Defensive Programming）实践，它实现了一个拦截器（Guardrail）

```python
class GuardrailFunctionOutput(BaseModel):
    # tripwire_triggered: bool (关键项)
    # 这个布尔值决定了“绊线”是否被触发。
    # 如果是 True，SDK 认为安检失败，会直接抛出错误或停止流程，拒绝执行发送工具。
    tripwire_triggered: bool 

    # output_info: dict (扩展项)
    # 这里存储安检员给出的“详细报告”。
    # 装进去传给系统，方便你在日志里查询为什么这封邮件被毙掉了。
    output_info: dict
```

In [31]:
from agents import output_guardrail, GuardrailFunctionOutput

In [37]:
# agent 参数是 SDK 协议要求, 本练习没有用到
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(starting_agent=checker_agent, input=message, context=ctx)

    review: EmailReview = result.final_output

    is_problem = review.is_contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(
        output_info={"review": review}, tripwire_triggered=is_problem
    )

In [38]:
# 这里刻意换一个人设, 测试下
cowboy_instructions = sales_manager_instructions + "\n像一个牛仔一样讲话"

cowboy_agent = Agent(
    name="cowboy agent",
    instructions=cowboy_instructions,
    output_guardrails=[email_guardrail],
    model=deepseek_model,
)

result = await Runner.run(starting_agent=cowboy_agent, input="写一封冷启动邮件")

logger.info(result.final_output)

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

$\color{yellow}{沙盒智能体}$

- 沙盒智能体可以使用专门的工具和 shell 命令来搜索并操作大型文档集、编辑文件、生成工件以及运行命令。
- 沙盒为模型提供了一个持久工作区，智能体可以用它代表你完成工作。Agents SDK 中的沙盒智能体可帮助你轻松运行与沙盒环境配对的智能体，便于将正确的文件放到文件系统上，并编排沙盒，从而轻松地大规模启动、停止和恢复任务。
- 你围绕智能体所需的数据来定义工作区。它可以从 GitHub 仓库、本地文件和目录、合成任务文件、S3 或 Azure Blob Storage 等远程文件系统，以及你提供的其他沙盒输入开始。

SandboxAgent 仍然是一个 Agent。它保留了常规的智能体接口，例如 instructions、prompt、tools、handoffs、mcp_servers、model_settings、output_type、安全防护措施以及 hooks，并且仍然通过普通的 Runner API 运行。变化在于执行边界：

- SandboxAgent 定义智能体本身：常规智能体配置，加上沙盒特定的默认值，例如 default_manifest、base_instructions、run_as，以及文件系统工具、shell 访问、技能、记忆或压缩等能力。
- Manifest 声明新沙盒工作区所需的初始内容和布局，包括文件、仓库、挂载和环境。
- 沙盒会话是命令运行且文件发生变化的实时隔离环境。
- SandboxRunConfig 决定本次运行如何获得该沙盒会话，例如直接注入一个会话、从序列化的沙盒会话状态重新连接，或通过沙盒客户端创建新的沙盒会话。
- 已保存的沙盒状态和快照让后续运行可以重新连接到之前的工作，或从已保存的内容为新的沙盒会话提供初始状态。

In [ ]:
code_dir = Path("code").resolve()  # 当前路径和 "code" 拼接
if not code_dir.exists():
    code_dir.mkdir()

output_dir = Path("output").resolve()  # 当前路径和 "output" 拼接
if not output_dir.exists():
    output_dir.mkdir()

In [ ]:
sandbox_agent_instructions = """你是一名专注于修复 Bug 的软件工程师。
请审阅沙盒 code 目录中的文件。
将修复后的文件版本写入到主机输出目录：
{output_dir}
写入文件时请务必使用完整的文件路径。
最后，请回复一份关于你所做修改的总结。
"""